# Reproduce the paper results

**Dhungana — Random versus Spatial Cross-Validation in Landslide Susceptibility Mapping (Far-Western Nepal)**

This notebook re-runs every experiment from the archived sample data. It does **not** need Google Earth Engine.

* Run the cells from top to bottom (**Runtime → Run all** also works after filling in Step 1).
* Total time: about 90 minutes on the free Colab CPU (most of it is Step 3).
* The last cell checks that every published number is reproduced.

## Step 1. Settings
Fill in **one** code source and **one** data source.

In [ ]:
# --- Code source: GitHub repository URL (leave empty to upload the code zip instead)
REPO_URL = ""          # e.g. "https://github.com/<username>/landslide_farwest_nepal"

# --- Data source A: Zenodo record ID with the processed data (leave empty to use Google Drive)
ZENODO_RECORD = ""     # e.g. "1234567"

# --- Data source B: folder in Google Drive with the backup files
DRIVE_BACKUP = "/content/drive/MyDrive/landslide_farwest/backup"

## Step 2. Get the code and the data

In [ ]:
import os, zipfile, glob, shutil, urllib.request
os.chdir("/content")
if REPO_URL:
    !git clone -q {REPO_URL} landslide_farwest_nepal
else:
    from google.colab import files
    up = files.upload()                      # choose landslide_farwest_nepal.zip
    z = [f for f in up if f.endswith('.zip')][0]
    zipfile.ZipFile(z).extractall('/content')
os.chdir('/content/landslide_farwest_nepal')
!pip -q install -r requirements.txt

PROC = 'data/processed'
os.makedirs(PROC, exist_ok=True)
NEEDED = ['samples.csv', 'dated_samples.csv', 'feature_names.json', 'features.tif']
if ZENODO_RECORD:
    for f in NEEDED:
        url = f'https://zenodo.org/records/{ZENODO_RECORD}/files/{f}?download=1'
        print('downloading', f); urllib.request.urlretrieve(url, f'{PROC}/{f}')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    for f in NEEDED:
        shutil.copy(f'{DRIVE_BACKUP}/{f}', f'{PROC}/{f}')
!ls -lh data/processed

## Step 3. Run the experiments (E1–E4, about 80 minutes)
Keep this tab open and the computer awake.

In [ ]:
!python 03_experiments.py

## Step 4. Susceptibility map (E5, about 10 minutes)
Downloads the inventory from Zenodo (needed for the boundary and landslide polygons).

In [ ]:
!python 02_prepare_data.py --download-only
!sed -i 's/MAP_MODEL = ".*"/MAP_MODEL = "XGBoost"/' config.py
!python 04_map.py

## Step 5. Check against the published values

In [ ]:
!python verify_results.py

## Step 6. Download the results

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download('results.zip')